# Notebook 01 — Base de conhecimento: reviews da Olist prontas para o RAG

Primeiro notebook do Tech Challenge 4 (*Voice of Customer Intelligence com RAG*).
Aqui transformamos os CSVs brutos da Olist em **uma tabela de documentos**: cada
linha é uma review com texto, já limpa, com os metadados que o RAG vai usar
para **citar a fonte** (`review_id`) e **filtrar** (nota, período, categoria,
estado, entrega).

**Como este notebook se relaciona com o código.** A lógica mora em
`src/voc_rag/` (testada com `pytest`); aqui nós **importamos** cada função,
**lemos o código dela** com `mostrar_codigo()` e **olhamos o efeito** nos dados.
Uma única fonte de verdade: se uma regra mudar, muda no `src/` e o notebook,
os testes e o `python -m voc_rag.preparacao` acompanham.

| Seção | Tema | Função em `src/voc_rag` | Aula de origem |
|---|---|---|---|
| 0 | Obter os dados do Kaggle | `dados.baixar_dados` | — |
| 1 | Carregar e inspecionar os CSVs | `dados.carregar_tabelas` | PLN 01 (corpus) |
| 2 | Seleção: só reviews com texto | `preparacao.selecionar_com_texto` | Requisito 1 do TC4 |
| 3 | Duplicatas de `review_id` | `preparacao.deduplicar_reviews` | Requisito 3 (rastreabilidade) |
| 4 | Junção dos metadados | `preparacao.categoria_por_pedido`, `juntar_metadados` | IA Gen 05 (chunk enriquecido) |
| 5 | Limpeza **leve** + PII | `texto.*`, `preparacao.limpar_textos` | PLN 02 · IA Gen 06 |
| 6 | Metadados derivados | `preparacao.derivar_metadados` | PLN 05 |
| 7 | Chunking: 1 review = 1 documento | — | IA Gen 02 |
| 8 | Validação e gravação em CSV | `preparacao.validar_base`, `salvar_base` | — |
| 9 | Prévia: a linha vira um `Document` | — | LangChain 03 |

**Saída:** `data/processed/base_conhecimento_reviews.csv`, lida pelo notebook 02.

> Execute as células **na ordem**. Pré-requisito, na raiz do repositório:
> `pip install -r requirements.txt` (instala o pacote `voc_rag` em modo editável).

## Seção 0 — Como obter os dados do Kaggle

O dataset é público: <https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce>
(licença CC BY-NC-SA 4.0). São 9 CSVs; usamos 6:

| Arquivo | Para quê |
|---|---|
| `olist_order_reviews_dataset.csv` | **o texto** (título, mensagem, nota, datas) |
| `olist_orders_dataset.csv` | status e datas do pedido (compra, entrega, prazo) |
| `olist_customers_dataset.csv` | estado e cidade do cliente |
| `olist_order_items_dataset.csv` | liga o pedido aos produtos (e preço/frete) |
| `olist_products_dataset.csv` | categoria do produto |
| `product_category_name_translation.csv` | nome da categoria em inglês (opcional) |

**Token do Kaggle:** em *Kaggle → Settings → API → Create New Token* você baixa o
`kaggle.json`. Coloque em `C:\Users\<voce>\.kaggle\kaggle.json` (Linux/Mac:
`~/.kaggle/kaggle.json`). Depois, de qualquer pasta:

```bash
python -m voc_rag.dados
```

A célula abaixo chama a mesma função, `baixar_dados()`, e só baixa se faltar algum CSV.
Sem token, baixe manualmente pelo botão *Download* do Kaggle e descompacte em `data/raw/`.

> **Por que os caminhos partem da raiz do repositório?** Um caminho relativo como
> `Path("data/raw")` é resolvido a partir da pasta em que o Jupyter ou o terminal foi
> aberto, não de onde o notebook está. E o `kagglehub`, sem `output_dir`, grava no
> cache do usuário (`~/.cache/kagglehub`), fora de qualquer repositório. Por isso
> `config.py` calcula a raiz a partir do próprio arquivo (`Path(__file__)`).

In [1]:
import inspect

import pandas as pd

from voc_rag import config, preparacao, texto
from voc_rag.dados import baixar_dados, carregar_tabelas

pd.set_option("display.max_colwidth", 120)


def mostrar_codigo(funcao):
    """Imprime o código-fonte de uma função do src/, para lermos junto com o resultado."""
    print(inspect.getsource(funcao))


PASTA_DADOS = baixar_dados()
print("Raiz do repositório:", config.RAIZ)
print("Lendo os CSVs de:   ", PASTA_DADOS)
print("Saída em:           ", config.ARQUIVO_BASE)

Raiz do repositório: C:\Development\1IAST-Fase4
Lendo os CSVs de:    C:\Development\1IAST-Fase4\data\raw
Saída em:            C:\Development\1IAST-Fase4\data\processed\base_conhecimento_reviews.csv


## Seção 1 — Carregar e inspecionar *(PLN aula 01: conhecer o corpus antes de mexer)*

Lemos as 6 tabelas e olhamos tamanho e colunas. **Observe** nas reviews quantas
linhas existem e quantas têm `review_comment_message` preenchido — essa diferença
é a primeira decisão de projeto da Seção 2.

In [2]:
tabelas = carregar_tabelas(PASTA_DADOS)

for chave, df in tabelas.items():
    print(f"{chave:9s} {df.shape[0]:>7,} linhas x {df.shape[1]:>2} colunas  ->  {list(df.columns)[:5]}...")

reviews = tabelas["reviews"]
print("\nValores ausentes nas reviews:")
print(reviews.isna().sum().to_string())

reviews    99,224 linhas x  7 colunas  ->  ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message']...
pedidos    99,441 linhas x  8 colunas  ->  ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at']...
clientes   99,441 linhas x  5 colunas  ->  ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']...
itens     112,650 linhas x  7 colunas  ->  ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date']...
produtos   32,951 linhas x  9 colunas  ->  ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty']...
traducao       71 linhas x  2 colunas  ->  ['product_category_name', 'product_category_name_english']...

Valores ausentes nas reviews:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_messag

In [3]:
# Três reviews ao acaso, para sentir o "jeito" do texto (gírias, \r\n, caixa alta...)
reviews.dropna(subset=["review_comment_message"]).sample(3, random_state=config.SEMENTE)[
    ["review_id", "review_score", "review_comment_title", "review_comment_message"]]

,review_id,review_score,review_comment_title,review_comment_message
37062,d3c62500417e7eeab7c6ac2151b67357,4,NaN,"Um produto como uma carteira somente poderia ser avaliado depois de algum tempo de uso, No entanto, o produto é bem ..."
17697,0f952ad4dfb88a77f12211a4038c70dd,5,muito bom produto,"Entrega no prazo , bom produto ."
41606,067b7c9c760d8bda7a2c38fb1f3fdf6e,5,NaN,entrega rápida


## Seção 2 — Seleção: só reviews com texto *(requisito 1 do TC4: ausentes)*

A maioria das reviews da Olist só tem a nota. Para um RAG, **uma review sem texto
não é evidência** — não há o que recuperar nem citar. Decisões:

1. **Texto = título + mensagem.** O título ("Recomendo", "Não chegou") carrega
   sentido; quando existir, vem antes da mensagem (`montar_texto`).
2. **Descartamos** reviews sem título e sem mensagem, e textos sem nenhuma letra
   (ex.: `"."`, `"..."`).
3. As reviews só com nota **não saem do projeto**: continuam valendo para
   estatísticas (ex.: distribuição de notas), só não entram no índice.

In [4]:
mostrar_codigo(preparacao.montar_texto)
mostrar_codigo(preparacao.selecionar_com_texto)

def montar_texto(titulo, mensagem):
    """Junta título e mensagem num único texto; ausentes viram string vazia."""
    partes = [str(p).strip() for p in (titulo, mensagem) if isinstance(p, str) and p.strip()]
    if len(partes) == 2 and not re.search(r"[.!?…]$", partes[0]):
        partes[0] += "."           # "Recomendo" + "Chegou rápido" -> "Recomendo. Chegou rápido"
    return " ".join(partes)

def selecionar_com_texto(reviews):
    """Fica só com as reviews cujo título + mensagem tem ao menos uma letra.

    Devolve (reviews com a coluna texto_bruto, contagens para o relatório).
    """
    reviews = reviews.copy()
    reviews["texto_bruto"] = [montar_texto(t, m) for t, m in
                              zip(reviews["review_comment_title"], reviews["review_comment_message"])]
    tem_letra = reviews["texto_bruto"].str.contains(r"[A-Za-zÀ-ÿ]", regex=True)
    contagens = {
        "total": len(reviews),
        "so_nota": int((reviews["texto_bruto"] == "").sum()),
        "sem_letr

In [5]:
com_texto, n = preparacao.selecionar_com_texto(reviews)
print(f"Reviews no total ............ {n['total']:>7,}")
print(f"Só nota (sem texto) ......... {n['so_nota']:>7,}  ({n['so_nota'] / n['total']:.0%})")
print(f"Texto sem nenhuma letra ..... {n['sem_letra']:>7,}")
print(f"Reviews que viram documento . {n['com_texto']:>7,}  ({n['com_texto'] / n['total']:.0%})")

Reviews no total ............  99,224
Só nota (sem texto) .........  56,537  (57%)
Texto sem nenhuma letra .....     233
Reviews que viram documento .  42,454  (43%)


## Seção 3 — Duplicatas de `review_id` *(requisito 3: rastreabilidade)*

O `review_id` é a **citação** que o RAG vai devolver, então precisa ser único na base.
Na Olist, o mesmo `review_id` aparece em mais de um pedido (o cliente comprou
várias coisas e respondeu uma pesquisa só). Se indexarmos as cópias, a mesma
evidência aparece duas vezes no top-k e conta em dobro nas perguntas agregadas.

**Decisão:** uma linha por `review_id`; os pedidos ficam juntos em `order_ids`
(separados por `|`) e os metadados vêm do primeiro pedido.

**Atenção:** textos iguais em reviews *diferentes* ("Recomendo") **não** são
duplicatas — são clientes diferentes dizendo a mesma coisa, e isso importa para
contar "quantos clientes reclamam de X".

In [6]:
mostrar_codigo(preparacao.deduplicar_reviews)

repetidos = com_texto["review_id"].duplicated(keep=False)
print(f"Linhas com review_id repetido: {repetidos.sum():,} "
      f"({com_texto.loc[repetidos, 'review_id'].nunique():,} review_ids distintos)")

unicas = preparacao.deduplicar_reviews(com_texto)
print(f"Depois da deduplicação: {len(unicas):,} reviews, review_id único? {unicas['review_id'].is_unique}")
print("\nTextos mais repetidos entre reviews DIFERENTES (mantidos de propósito):")
print(unicas["texto_bruto"].str.lower().value_counts().head(5).to_string())

def deduplicar_reviews(reviews):
    """Uma linha por review_id; os pedidos da mesma review ficam em order_ids ("a|b")."""
    pedidos_por_review = (reviews.groupby("review_id")["order_id"]
                          .agg(lambda ids: "|".join(sorted(set(ids)))).rename("order_ids"))
    return (reviews.sort_values(["review_id", "order_id"])
            .drop_duplicates("review_id", keep="first")
            .merge(pedidos_por_review, on="review_id")
            .reset_index(drop=True))

Linhas com review_id repetido: 622 (306 review_ids distintos)


Depois da deduplicação: 42,138 reviews, review_id único? True

Textos mais repetidos entre reviews DIFERENTES (mantidos de propósito):
texto_bruto
muito bom    446
bom          377
recomendo    343
ótimo        266
otimo        235


## Seção 4 — Junção dos metadados *(IA Gen aula 05: o chunk enriquecido)*

O bônus do TC4 pede filtros por nota, período, pedido, entrega e categoria. Esses
campos estão espalhados em outras tabelas:

```
reviews ──order_id──> pedidos ──customer_id──> clientes   (estado, cidade)
   └──order_id──> itens ──product_id──> produtos (categoria)
```

Um pedido pode ter **vários itens de categorias diferentes**. Decisão: a
`categoria` do documento é a **mais frequente** entre os itens do pedido (empate:
a do primeiro item); guardamos também `n_categorias` para saber quando a review
pode estar falando de mais de um produto. Categoria ausente vira `"sem_categoria"`.

Todas as junções são *left join*: nenhuma review se perde por falta de metadado,
e `juntar_metadados` confere que nenhuma foi duplicada.

In [7]:
mostrar_codigo(preparacao.categoria_por_pedido)

categorias = preparacao.categoria_por_pedido(tabelas["itens"], tabelas["produtos"], tabelas["traducao"])
base = preparacao.juntar_metadados(unicas, tabelas["pedidos"], tabelas["clientes"], categorias)

print(f"{len(base):,} reviews depois das junções (igual à Seção 3: {len(unicas):,})")
print("\nAusentes nos metadados:")
print(base[["order_status", "order_delivered_customer_date", "customer_state", "categoria_en"]]
      .isna().sum().to_string())
print("\nCategorias mais comuns:")
print(base["categoria"].value_counts().head(8).to_string())

def categoria_por_pedido(itens, produtos, traducao):
    """Categoria mais frequente entre os itens do pedido (empate: a do primeiro item)."""
    itens = itens.sort_values(["order_id", "order_item_id"])
    itens_cat = (itens.merge(produtos[["product_id", "product_category_name"]], on="product_id", how="left")
                 .merge(traducao, on="product_category_name", how="left"))
    itens_cat["product_category_name"] = itens_cat["product_category_name"].fillna("sem_categoria")
    return itens_cat.groupby("order_id", sort=False).agg(
        categoria=("product_category_name", lambda c: c.value_counts().index[0]),
        categoria_en=("product_category_name_english",
                      lambda c: c.dropna().value_counts().index[0] if c.notna().any() else None),
        n_categorias=("product_category_name", "nunique"),
        n_itens=("order_item_id", "count"),
        valor_produtos=("price", "sum"),
        valor_frete=("freight_value", "sum"),
    ).reset_index()



42,138 reviews depois das junções (igual à Seção 3: 42,138)

Ausentes nos metadados:
order_status                        0
order_delivered_customer_date    1864
customer_state                      0
categoria_en                     1111

Categorias mais comuns:
categoria
cama_mesa_banho           4319
beleza_saude              3508
esporte_lazer             3013
informatica_acessorios    2779
moveis_decoracao          2589
relogios_presentes        2571
utilidades_domesticas     2465
telefonia                 1885


## Seção 5 — Limpeza **leve** do texto *(PLN aula 02 · IA Gen aula 06)*

Na PLN aula 02, `limpar_normalizar()` deixa tudo minúsculo, expande contrações e
pode tirar acentos. O slide 18 avisa: **a limpeza é uma decisão sensível ao
downstream.** Por isso geramos **duas versões** do texto:

| Coluna | Para quem | O que faz |
|---|---|---|
| `texto` | embeddings, LLM e **a citação mostrada ao usuário** | `limpar_leve`: só tira lixo de formatação (`\r\n`, espaços repetidos) e mascara PII. **Mantém** caixa, acentos, pontuação e emojis — o modelo de embeddings foi treinado com texto natural |
| `texto_lexical` | BM25 (notebook 02) | `limpar_normalizar()` da PLN 02 **sem acento** — o cliente escreve "nao" e "não", e a busca por palavra precisa casar os dois |

Não fazemos stemming nem tiramos stop words aqui: "não" é stop word em listas
comuns e inverte o sentido de "não chegou". O BM25 decide isso na tokenização dele.

**PII (IA Gen aula 06):** algumas reviews trazem telefone ou e-mail do cliente.
Mascaramos antes de indexar — o que entra no índice pode aparecer numa resposta.

In [8]:
mostrar_codigo(texto.limpar_leve)
mostrar_codigo(texto.limpar_normalizar)
mostrar_codigo(texto.mascarar_pii)
print(texto.PADROES_PII)

def limpar_leve(texto):
    """Para embeddings/LLM: só formatação. Caixa, acentos e pontuação ficam."""
    texto = texto.replace("\\r\\n", " ").replace("\\n", " ")  # \r\n escrito como texto
    texto = re.sub(r"\s+", " ", texto)                        # \r\n reais, tabs, espaços repetidos
    return texto.strip()

def limpar_normalizar(texto, remove_acentos=False):
    """Pipeline modular para texto em PT-BR (PLN aula 02, slide 20)."""
    if not isinstance(texto, str):
        return ""
    texto = texto.lower()                                            # 1. minúsculas
    texto = re.sub(r"<[^>]+>", " ", texto)                           # 2. HTML
    texto = re.sub(r"http\S+|www\.\S+", " ", texto)                  # 3. URLs
    texto = re.sub(r"\S+@\S+", " ", texto)                           # 4. e-mails
    texto = " ".join(CONTRACTIONS.get(p, p) for p in texto.split())  # 5. contrações
    texto = re.sub(r"[^a-záàâãäéèêëíìîïóòôõöúùûüç\s.,!?]", " ", texto)  # 6. só letras e pontua

In [9]:
# Teste rápido com uma frase "suja", antes de aplicar na base toda
sujo = "N chegou ainda!!\r\nVc pode me ligar no (11) 98877-1234?"
print("leve + PII :", texto.mascarar_pii(texto.limpar_leve(sujo)))
print("lexical    :", texto.limpar_normalizar(sujo, remove_acentos=True))

leve + PII : ('N chegou ainda!! Vc pode me ligar no [TELEFONE]?', ['TELEFONE'])
lexical    : nao chegou ainda!! voce pode me ligar no ?


In [10]:
base = preparacao.limpar_textos(base)

print("Reviews com PII mascarada:", (base["pii_mascarada"] != "").sum())
print(base.loc[base["pii_mascarada"] != "", ["texto", "pii_mascarada"]].head(3).to_string())

print("\nANTES x DEPOIS (3 exemplos):")
for _, linha in base.sample(3, random_state=config.SEMENTE).iterrows():
    print(f"  bruto   : {linha['texto_bruto'][:110]!r}")
    print(f"  texto   : {linha['texto'][:110]!r}")
    print(f"  lexical : {linha['texto_lexical'][:110]!r}\n")

Reviews com PII mascarada: 2
                                                                                                                             texto pii_mascarada
19280  Aguardo o recebimento da minha encomenda com urgência... No momento estou Insatisfeita com a empresa. [TELEFONE] [TELEFONE]      TELEFONE
40894                                                                                     Faltou item descrito Me ligue [TELEFONE]      TELEFONE

ANTES x DEPOIS (3 exemplos):
  bruto   : 'Comprei 4 kit manicure e só recebi 1.'
  texto   : 'Comprei 4 kit manicure e só recebi 1.'
  lexical : 'comprei kit manicure e so recebi .'

  bruto   : 'Muito feliz com a compra, amei o relógio'
  texto   : 'Muito feliz com a compra, amei o relógio'
  lexical : 'muito feliz com a compra, amei o relogio'

  bruto   : 'super recomendo!!! Chegou antes do prazo...'
  texto   : 'super recomendo!!! Chegou antes do prazo...'
  lexical : 'super recomendo!!! chegou antes do prazo...'



**Audite a regex de PII.** Leia as reviews mascaradas acima: um número de pedido
ou de rastreio pode cair no padrão de telefone (falso positivo). Na aula 06 isso se
resolve com validação aritmética; aqui, como o custo de mascarar a mais é baixo
(perdemos um número, não o sentido), aceitamos o falso positivo.

## Seção 6 — Metadados derivados *(PLN aula 05: enriquecer a review)*

Campos que o RAG vai usar como **filtro** e que o LLM pode citar:

- `sentimento_nota`: 1–2 → `negativo`, 3 → `neutro`, 4–5 → `positivo`. É um rótulo
  **barato e confiável** (o próprio cliente deu a nota). Na PLN 05 treinamos um
  classificador sobre o texto; ele pode entrar depois para achar reviews em que
  texto e nota discordam.
- `dias_entrega` e `atraso_dias` (entrega real − prazo estimado) e
  `entregue_com_atraso`: respondem "o que dizem os clientes cujo pedido atrasou?".
  Pedido não entregue fica com `entregue_com_atraso` ausente, não `False`.
- `ano_mes`: filtro de período ("reclamações em 2018-03").
- `n_palavras`: ajuda a filtrar reviews curtas demais para servirem de evidência.

In [11]:
mostrar_codigo(preparacao.derivar_metadados)

base = preparacao.derivar_metadados(base)
print(pd.crosstab(base["sentimento_nota"], base["entregue_com_atraso"].astype(str),
                  normalize="columns").round(2))
print("\nCompare as colunas: se a fatia de negativas for bem maior entre os atrasados,")
print("a entrega é candidata a 'principal problema' (pergunta agregada do TC4).")

def derivar_metadados(base):
    """Sentimento pela nota, prazos de entrega, período e tamanho do texto."""
    base = base.copy()
    for col in ["review_creation_date", "review_answer_timestamp", "order_purchase_timestamp",
                "order_delivered_customer_date", "order_estimated_delivery_date"]:
        base[col] = pd.to_datetime(base[col], errors="coerce")

    base["sentimento_nota"] = pd.cut(base["review_score"], bins=[0, 2, 3, 5],
                                     labels=["negativo", "neutro", "positivo"]).astype(str)
    base["dias_entrega"] = (base["order_delivered_customer_date"]
                            - base["order_purchase_timestamp"]).dt.days
    base["atraso_dias"] = (base["order_delivered_customer_date"].dt.normalize()
                           - base["order_estimated_delivery_date"].dt.normalize()).dt.days
    base["entregue_com_atraso"] = (base["atraso_dias"] > 0).astype("boolean")
    base.loc[base["atraso_dias"].isna(), "entregue_com_atraso"] = pd.N

## Seção 7 — Chunking: por que 1 review = 1 documento *(IA Gen aula 02)*

Na aula 02, `dividir_em_chunks()` corta documentos longos em pedaços de ~400
caracteres com overlap. Aqui o documento **já é curto**: veja a distribuição de
tamanho abaixo. Cortar uma review no meio separaria a reclamação do motivo e
duplicaria a citação. **Decisão: cada review é um chunk inteiro**, e o
`review_id` funciona também como `chunk_id`.

E o *chunk enriquecido* da aula 05 (colocar "Nota 1 · Categoria: cama_mesa_banho"
dentro do texto)? Deixamos os metadados **fora** do texto que vira embedding e os
usamos como filtro. Colocá-los no texto é uma hipótese a testar no protocolo de
avaliação, não o ponto de partida.

In [12]:
tamanho = base["texto"].str.len()
print(tamanho.describe(percentiles=[0.5, 0.9, 0.99]).round(0).to_string())
print(f"\nReviews acima de {config.LIMITE_CHUNK} caracteres: {(tamanho > config.LIMITE_CHUNK).sum():,} "
      f"({(tamanho > config.LIMITE_CHUNK).mean():.2%})")
print(f"Reviews com 1 ou 2 palavras (evidência fraca, mas mantidas): {(base['n_palavras'] <= 2).sum():,}")

count    42138.0
mean        69.0
std         55.0
min          1.0
50%         54.0
90%        162.0
99%        210.0
max        229.0

Reviews acima de 400 caracteres: 0 (0.00%)
Reviews com 1 ou 2 palavras (evidência fraca, mas mantidas): 5,983


## Seção 8 — Validação e gravação em CSV

Antes de gravar, `validar_base` checa as garantias que o resto do pipeline assume —
se uma delas quebrar, é melhor descobrir aqui do que numa resposta errada do RAG.

Gravamos em **CSV** com `pandas` (UTF-8, sem índice). Datas vão como texto ISO
(`2018-03-15 00:00:00`); `carregar_base` as relê como datas.

> **Sem o notebook:** `python -m voc_rag.preparacao` roda as Seções 2 a 8 de uma vez
> (função `construir_base`) e grava o mesmo CSV. É o caminho reprodutível para
> regenerar a base; o notebook é o caminho para entender.

In [13]:
mostrar_codigo(preparacao.validar_base)

documentos = preparacao.selecionar_colunas(base)
preparacao.validar_base(documentos)
print("Validações OK.")

caminho = preparacao.salvar_base(documentos)
print(f"Gravado: {caminho}  ({len(documentos):,} documentos x {documentos.shape[1]} colunas, "
      f"{caminho.stat().st_size / 1e6:.1f} MB)")

def validar_base(documentos):
    """Garantias que o resto do pipeline assume. Levanta ValueError se alguma quebrar."""
    checagens = {
        "review_id duplicado: a citação ficaria ambígua": documentos["review_id"].is_unique,
        "documento sem texto": documentos["texto"].str.len().gt(0).all(),
        "nota fora de 1..5": documentos["nota"].between(1, 5).all(),
        "quebra de linha sobrou no texto":
            not documentos["texto"].str.contains(r"\\r\\n|\r|\n", regex=True).any(),
    }
    falhas = [msg for msg, ok in checagens.items() if not ok]
    if falhas:
        raise ValueError("; ".join(falhas))

Validações OK.


Gravado: C:\Development\1IAST-Fase4\data\processed\base_conhecimento_reviews.csv  (42,138 documentos x 28 colunas, 20.4 MB)


In [14]:
# Releitura: é assim que o notebook 02 vai abrir a base
relida = preparacao.carregar_base()
assert len(relida) == len(documentos) and relida["review_id"].is_unique

# Prova de que o notebook e o pipeline de linha de comando geram a MESMA base
pelo_pipeline = preparacao.construir_base(tabelas)
pd.testing.assert_frame_equal(pelo_pipeline.reset_index(drop=True), documentos.reset_index(drop=True))
print("Notebook e construir_base() geram a mesma base.")
relida.head(3)

Notebook e construir_base() geram a mesma base.


,review_id,order_ids,texto,texto_lexical,titulo_original,mensagem_original,nota,sentimento_nota,data_review,data_resposta,...,categoria,categoria_en,n_categorias,n_itens,valor_produtos,valor_frete,estado,cidade,n_palavras,pii_mascarada
0,00020c7512a52e92212f12d3e37513c0,e28abf2eb2f1fbcbdc2dd0cd9a561671,Entrega rápida! A entrega foi super rápida e o pendente é lindo! Igual a foto mesmo!,entrega rapida! a entrega foi super rapida e o pendente e lindo! igual a foto mesmo!,Entrega rápida!,A entrega foi super rápida e o pendente é lindo! Igual a foto mesmo!,5,positivo,2018-04-25 00:00:00,2018-04-26 14:55:36,...,casa_construcao,home_construction,1.0,1.0,150.00,17.96,SP,sao jose do rio preto,16,
1,00046a69550325aea5fb89f65c7387f2,9fbda7367628952bc36c3512c46d887b,"SUPER RECOMENDO. GOSTEI DA CAPINHA DE CELULAR, VEIO COMO EU ESPERAVA.","super recomendo. gostei da capinha de celular, veio como eu esperava.",SUPER RECOMENDO,"GOSTEI DA CAPINHA DE CELULAR, VEIO COMO EU ESPERAVA.",5,positivo,2018-07-01 00:00:00,2018-07-02 12:32:24,...,telefonia,telephony,1.0,1.0,9.99,15.23,RJ,rio de janeiro,11,
2,0005534973388c830bb858cfba83b17b,a589caa6892ceacc6bbf2f8cc30a8ad4,otimo produto. prazo cumprido. sabor tambem muito bom!,otimo produto. prazo cumprido. sabor tambem muito bom!,NaN,otimo produto. prazo cumprido. sabor tambem muito bom!,5,positivo,2017-09-01 00:00:00,2017-09-02 13:04:08,...,esporte_lazer,sports_leisure,1.0,1.0,109.90,30.30,AL,maceio,8,


### Dicionário de campos (para o README)

| Campo | Uso no RAG |
|---|---|
| `review_id` | **citação** e id do chunk |
| `order_ids` | rastrear até o pedido (mais de um quando a review foi compartilhada) |
| `texto` | entra no embedding e é a evidência mostrada ao usuário |
| `texto_lexical` | entra no BM25 |
| `titulo_original`, `mensagem_original` | auditoria: o texto exatamente como veio da Olist |
| `nota`, `sentimento_nota` | filtro ("só reclamações") |
| `data_review`, `ano_mes` | filtro de período |
| `status_pedido`, `dias_entrega`, `atraso_dias`, `entregue_com_atraso` | filtro e contexto de entrega |
| `categoria`, `categoria_en`, `n_categorias` | filtro por categoria de produto |
| `estado`, `cidade` | filtro por região |
| `n_palavras`, `pii_mascarada` | qualidade da evidência e auditoria de LGPD |

## Seção 9 — Prévia: a linha vira um `Document` *(LangChain aula 03)*

No notebook 02, cada linha vira um `Document` do LangChain: `page_content` é o
`texto` e os metadados vão no dicionário `metadata` — é por ele que o FAISS
filtra e que a resposta cita o `review_id`. Aqui só mostramos o formato, como
dicionário; a conversão de verdade entra no `src/` quando montarmos o pipeline RAG.

In [15]:
METADADOS = ["review_id", "nota", "sentimento_nota", "ano_mes", "categoria", "estado",
             "entregue_com_atraso", "atraso_dias"]
exemplo = relida[relida["sentimento_nota"] == "negativo"].iloc[0]
{"page_content": exemplo["texto"],
 "metadata": {c: (None if pd.isna(exemplo[c]) else exemplo[c]) for c in METADADOS}}

{'page_content': 'O produto chegou no prazo porém o item veio errado, entrei em contato com o vendedor e o mesmo disse que me mandaria o produto correto. Estou aguardando o código de rastreio do produto correto.',
 'metadata': {'review_id': '00125f12c201cf0160f4a1457b466512',
  'nota': np.int64(2),
  'sentimento_nota': 'negativo',
  'ano_mes': '2018-03',
  'categoria': 'artes',
  'estado': 'SP',
  'entregue_com_atraso': False,
  'atraso_dias': np.float64(-4.0)}}

## Resumo das decisões

| Decisão | Por quê |
|---|---|
| Só reviews com texto viram documento | review sem texto não é evidência recuperável nem citável |
| Título + mensagem num texto só | o título carrega sentido ("Não chegou") |
| Uma linha por `review_id` | a citação precisa ser única; cópias contariam em dobro |
| Textos iguais em reviews diferentes ficam | são clientes diferentes; importam para contagens |
| Limpeza leve para embedding, normalização só no `texto_lexical` | embeddings gostam de texto natural; BM25 precisa casar "nao" e "não" |
| PII mascarada antes de indexar | o que está no índice pode sair na resposta (LGPD) |
| Sem chunking | reviews são curtas; cortar separaria reclamação e motivo |
| Metadados fora do texto, como filtro | chunk enriquecido fica como hipótese a avaliar |
| CSV com pandas | simples de abrir e reler no notebook 02 |
| Lógica em `src/`, notebook só explica | uma fonte de verdade, testada e reprodutível |

**Próximo:** `02_indice.ipynb` — embeddings com `paraphrase-multilingual-MiniLM-L12-v2`
e índice FAISS sobre esta base.